# Ensemble + Abstention Model

Final model for the concrete delamination tap-test classifier.

## Approach

1. **All 31 training pillars** (good 1–13, bad 1–18) — no exclusions.
2. **6 held-out pillars** (good 14–16, bad 19–21) — *never* used for training, threshold tuning, or model selection.
3. **Per-clip CMVN** on the mel spectrogram — removes channel response, makes recordings made under different conditions look more similar.
4. **Waveform augmentation ×5** — volume jitter, noise injection, onset jitter, pitch shift. Trains the model to be invariant to recording conditions.
5. **Rich features (396-dim)** — mel mean+std, CMVN-mel mean+std, delta-mel mean+std, decay time τ, spectral shape descriptors (centroid, spread, skewness, kurtosis, rolloff85/95, flatness, slope), temporal descriptors (rise time, ZCR, envelope kurtosis).
6. **Three base classifiers** — RBF-SVM, Gradient Boosting, Random Forest. Each captures a different pattern.
7. **Ensemble** — average of `P(bad)` from the three models.
8. **Per-clip abstention zone** — clips with `P(bad) ∈ [0.4, 0.6]` don't vote. Pillar verdict is the majority of confident clips. Borderline clips trigger "knock again" in deployment.

## Held-out result

**6/6 pillars correctly classified** on data the model never saw.

In [1]:
import os, glob, warnings, pickle, time
import numpy as np
import librosa
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

# Project root — auto-detected by walking up until data/ and model/ are found.
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'model'))):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError('Project root not found (no data/ and model/ above current directory)')
    ROOT = parent
SR, WIN, HOP, N_MELS, FMAX, ONSET = 16000, 3200, 64, 64, 8000, 0.40

TRAIN_GOOD   = list(range(1, 14))   # 1..13
TRAIN_BAD    = list(range(1, 19))   # 1..18
HOLDOUT_GOOD = [14, 15, 16]
HOLDOUT_BAD  = [19, 20, 21]
N_AUG = 4                            # 4 augmented variants + 1 original = 5 per clip
ABSTENTION_LO, ABSTENTION_HI = 0.40, 0.60

rng = np.random.default_rng(42)
print(f'Training pillars: {len(TRAIN_GOOD)+len(TRAIN_BAD)} ({len(TRAIN_GOOD)} good, {len(TRAIN_BAD)} bad)')
print(f'Holdout pillars : {len(HOLDOUT_GOOD)+len(HOLDOUT_BAD)} (never trained on)')
print(f'Abstention zone : P(bad) ∈ [{ABSTENTION_LO}, {ABSTENTION_HI}]')

Training pillars: 31 (13 good, 18 bad)
Holdout pillars : 6 (never trained on)
Abstention zone : P(bad) ∈ [0.4, 0.6]


## Feature extraction

Each 200ms knock window is encoded as a 396-dim feature vector covering:
- Spectral content (mel mean + std)
- Channel-invariant spectral shape (CMVN-normalized mel mean + std)
- Temporal evolution (delta-mel mean + std)
- Physically motivated decay time τ
- Spectral shape descriptors and temporal descriptors

In [2]:
def find_onset(y, thresh=ONSET):
    p = np.max(np.abs(y))
    if p < 1e-8: return None, 0.0
    above = np.where(np.abs(y) >= thresh * p)[0]
    if len(above) == 0: return None, p
    return above[0], p

def window_around(y, onset):
    half = WIN // 2
    start, end = onset - half, onset + half
    clip = np.zeros(WIN, dtype=np.float32)
    s, e = max(0, start), min(len(y), end)
    d = s - start
    clip[d:d + (e - s)] = y[s:e]
    return clip

def cmvn(Sdb):
    """Cepstral mean+variance normalization — removes channel response."""
    mu = Sdb.mean(axis=1, keepdims=True)
    sd = Sdb.std(axis=1, keepdims=True) + 1e-6
    return (Sdb - mu) / sd

def decay_time(clip):
    """Fit exponential decay to post-onset RMS envelope. Intact concrete rings longer."""
    on, peak = find_onset(clip)
    if on is None: return 0.0
    win = 80
    env = np.array([np.sqrt(np.mean(clip[i:i+win]**2)) for i in range(on, len(clip)-win, win)])
    if len(env) < 4 or env.max() < 1e-6: return 0.0
    env = env / env.max()
    log_env = np.log(env + 1e-6)
    t = np.arange(len(env)) * win / SR
    try:
        slope, _ = np.polyfit(t, log_env, 1)
        return float(-1.0 / slope) if slope < 0 else 0.0
    except Exception:
        return 0.0

def spectral_descriptors(clip):
    S = np.abs(librosa.stft(clip, n_fft=512, hop_length=HOP))
    f = librosa.fft_frequencies(sr=SR, n_fft=512)
    mag = S.mean(axis=1) + 1e-9
    mag /= mag.sum()
    centroid = float(np.sum(f * mag))
    spread   = float(np.sqrt(np.sum(((f - centroid)**2) * mag)))
    skewness = float(np.sum(((f - centroid)**3) * mag) / (spread**3 + 1e-9))
    kurt     = float(np.sum(((f - centroid)**4) * mag) / (spread**4 + 1e-9))
    cum = np.cumsum(mag)
    rolloff85 = float(f[np.searchsorted(cum, 0.85)])
    rolloff95 = float(f[np.searchsorted(cum, 0.95)])
    flatness = float(librosa.feature.spectral_flatness(S=S).mean())
    slope    = float(np.polyfit(f, np.log(mag + 1e-9), 1)[0])
    return [centroid, spread, skewness, kurt, rolloff85, rolloff95, flatness, slope]

def temporal_descriptors(clip):
    on, peak = find_onset(clip)
    if on is None: return [0.0, 0.0, 0.0]
    abs_clip = np.abs(clip)
    third = peak / 3.0
    above_third = np.where(abs_clip >= third)[0]
    rise_samples = (on - above_third[0]) if len(above_third) and above_third[0] < on else 0
    rise_time = rise_samples / SR
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(clip, hop_length=HOP)))
    win = 80
    env = np.array([np.sqrt(np.mean(clip[i:i+win]**2)) for i in range(0, len(clip)-win, win)])
    if env.max() > 1e-6:
        env_norm = env / env.max()
        env_kurt = float(((env_norm - env_norm.mean())**4).mean() / (env_norm.std()**4 + 1e-9))
    else:
        env_kurt = 0.0
    return [rise_time, zcr, env_kurt]

def features_from_waveform(y):
    """Full 396-dim feature vector from a waveform. Returns None on failure."""
    on, peak = find_onset(y)
    if on is None: return None
    y = y / peak
    on, _ = find_onset(y)
    clip = window_around(y, on)
    Sdb = librosa.power_to_db(
        librosa.feature.melspectrogram(y=clip, sr=SR, hop_length=HOP, n_mels=N_MELS, fmax=FMAX),
        ref=np.max)
    Sdb_n = cmvn(Sdb)
    f_mel      = np.concatenate([Sdb.mean(axis=1),   Sdb.std(axis=1)])
    f_mel_cmvn = np.concatenate([Sdb_n.mean(axis=1), Sdb_n.std(axis=1)])
    d_mel      = librosa.feature.delta(Sdb)
    f_dmel     = np.concatenate([d_mel.mean(axis=1), d_mel.std(axis=1)])
    return np.concatenate([f_mel, f_mel_cmvn, f_dmel,
                            [decay_time(clip)],
                            spectral_descriptors(clip),
                            temporal_descriptors(clip)])

def pillar_id_from_name(name):
    base = name.split('-')[0]
    return int(''.join(c for c in base if c.isdigit()))

print(f'Feature dim = {128+128+128+1+8+3} = 396')

Feature dim = 396 = 396


## Augmentation pipeline

Each training clip is augmented into 4 variants by:
- volume jitter (±6 dB) — simulates different mic distances
- pink-ish noise injection (SNR 20–35 dB) — simulates different rooms
- onset jitter (±10 ms) — simulates inconsistent windowing
- pitch shift (±0.4 semitones) — simulates resonance variation

Holdout clips are *never* augmented.

In [3]:
def aug_volume(y):
    return y * 10 ** (rng.uniform(-6.0, 6.0) / 20.0)

def aug_noise(y):
    snr = rng.uniform(20.0, 35.0)
    sig_pow = np.mean(y**2) + 1e-12
    noise_pow = sig_pow / (10 ** (snr / 10))
    return y + rng.standard_normal(len(y)).astype(np.float32) * np.sqrt(noise_pow)

def aug_onset_jitter(y):
    shift = int(rng.uniform(-10.0, 10.0) / 1000.0 * SR)
    if shift > 0: return np.concatenate([np.zeros(shift, dtype=y.dtype), y[:-shift]])
    if shift < 0: return np.concatenate([y[-shift:], np.zeros(-shift, dtype=y.dtype)])
    return y

def aug_pitch(y):
    return librosa.effects.pitch_shift(y=y, sr=SR, n_steps=rng.uniform(-0.4, 0.4))

AUGS = [aug_volume, aug_noise, aug_onset_jitter, aug_pitch]

def augment_variants(y, n=N_AUG):
    return [AUGS[i % len(AUGS)](y).astype(np.float32) for i in range(n)]

print(f'{N_AUG} augmentation variants per training clip')

4 augmentation variants per training clip


## Load and feature-extract

Training data is augmented; holdout data is not. Features cached to disk so re-runs are fast (~5s with cache, ~2 min without).

In [4]:
CACHE = os.path.join(ROOT, 'model', 'robust_v1_features_cache.pkl')

def load_train_set():
    rows = []
    for f in sorted(glob.glob(os.path.join(ROOT, 'data', 'good', '*.wav'))):
        pp = pillar_id_from_name(os.path.basename(f))
        if pp not in TRAIN_GOOD: continue
        y, _ = librosa.load(f, sr=SR, mono=True)
        feat = features_from_waveform(y)
        if feat is None: continue
        rows.append((feat, 0, f'good_{pp}', 0))
        for vi, ya in enumerate(augment_variants(y), start=1):
            fa = features_from_waveform(ya)
            if fa is not None: rows.append((fa, 0, f'good_{pp}', vi))
    for f in sorted(glob.glob(os.path.join(ROOT, 'data', 'bad', '*.wav'))):
        pp = pillar_id_from_name(os.path.basename(f))
        if pp not in TRAIN_BAD: continue
        y, _ = librosa.load(f, sr=SR, mono=True)
        feat = features_from_waveform(y)
        if feat is None: continue
        rows.append((feat, 1, f'bad_{pp}', 0))
        for vi, ya in enumerate(augment_variants(y), start=1):
            fa = features_from_waveform(ya)
            if fa is not None: rows.append((fa, 1, f'bad_{pp}', vi))
    X = np.array([r[0] for r in rows]); y = np.array([r[1] for r in rows])
    pid = np.array([r[2] for r in rows]); var = np.array([r[3] for r in rows])
    return X, y, pid, var

def load_holdout_set():
    rows = []
    for sub in sorted(os.listdir(os.path.join(ROOT, 'data', 'holdout'))):
        sub_path = os.path.join(ROOT, 'data', 'holdout', sub)
        if not os.path.isdir(sub_path): continue
        truth = 1 if sub.startswith('bad') else 0
        for f in sorted(glob.glob(os.path.join(sub_path, '*.wav'))):
            y, _ = librosa.load(f, sr=SR, mono=True)
            feat = features_from_waveform(y)
            if feat is not None: rows.append((feat, truth, sub, 0))
    X = np.array([r[0] for r in rows]); y = np.array([r[1] for r in rows])
    pid = np.array([r[2] for r in rows]); var = np.array([r[3] for r in rows])
    return X, y, pid, var

if os.path.exists(CACHE):
    Xtr, ytr, ptr, vtr, Xho, yho, pho, vho = pickle.load(open(CACHE, 'rb'))
    print(f'Loaded features from cache.')
else:
    print('Extracting training features (with augmentation)...')
    t0 = time.time()
    Xtr, ytr, ptr, vtr = load_train_set()
    print(f'  {len(Xtr)} rows in {time.time()-t0:.1f}s')
    print('Extracting holdout features (no augmentation)...')
    t0 = time.time()
    Xho, yho, pho, vho = load_holdout_set()
    print(f'  {len(Xho)} rows in {time.time()-t0:.1f}s')
    pickle.dump((Xtr, ytr, ptr, vtr, Xho, yho, pho, vho), open(CACHE, 'wb'))
    print('Cached.')

# Verify holdout discipline
overlap = set(ptr.tolist()) & set(pho.tolist())
assert not overlap, f'LEAKAGE: pillars in both sets: {overlap}'
print()
print(f'Training : {len(Xtr)} rows from {len(set(ptr))} pillars')
print(f'Holdout  : {len(Xho)} rows from {len(set(pho))} pillars  (disjoint ✓)')
print(f'Feature dim : {Xtr.shape[1]}')

Extracting training features (with augmentation)...


  2540 rows in 17.7s
Extracting holdout features (no augmentation)...
  72 rows in 0.2s
Cached.

Training : 2540 rows from 31 pillars
Holdout  : 72 rows from 6 pillars  (disjoint ✓)
Feature dim : 396


## Step 1 — LOPO on training pillars

31 leave-one-pillar-out folds. Each model is trained on 30 pillars (with all augmented variants) and predicts on the original clips of the held-out pillar.

This gives an honest within-distribution estimate. SVM uses sigmoid on `decision_function` for speed (no internal Platt CV per fold).

In [5]:
TRAIN_PILLARS = sorted(set(ptr))

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

model_specs = [
    ('SVM', lambda: SVC(kernel='rbf', C=5.0, gamma=0.001,
                        class_weight='balanced', probability=False)),
    ('GBM', lambda: GradientBoostingClassifier(n_estimators=100, max_depth=2,
                                                learning_rate=0.1, random_state=42)),
    ('RF',  lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                            n_jobs=-1, random_state=42)),
]

# Subsample augmented variants for LOPO speed (original + first 2 augs)
keep = vtr <= 2
Xt_l, yt_l, pt_l, vt_l = Xtr[keep], ytr[keep], ptr[keep], vtr[keep]

lopo_probs = {name: np.full(len(Xt_l), np.nan) for name, _ in model_specs}
t0 = time.time()
for tp_i, tp in enumerate(TRAIN_PILLARS):
    test_mask  = (pt_l == tp) & (vt_l == 0)
    train_mask = pt_l != tp
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xt_l[train_mask])
    Xte_s = sc.transform(Xt_l[test_mask])
    for name, builder in model_specs:
        clf = builder()
        clf.fit(Xtr_s, yt_l[train_mask])
        if isinstance(clf, SVC):
            lopo_probs[name][test_mask] = sigmoid(clf.decision_function(Xte_s))
        else:
            lopo_probs[name][test_mask] = clf.predict_proba(Xte_s)[:, 1]
    if (tp_i + 1) % 10 == 0:
        print(f'  {tp_i+1}/{len(TRAIN_PILLARS)} folds in {time.time()-t0:.0f}s')

orig_mask = vt_l == 0
y_orig = yt_l[orig_mask]
p_orig = pt_l[orig_mask]

def pillar_acc(probs, labels, pids, threshold=0.5):
    ok = total = 0
    for p in sorted(set(pids)):
        m = pids == p
        votes = (probs[m] >= threshold).astype(int)
        vote = int(np.bincount(votes, minlength=2).argmax())
        ok += int(vote == int(labels[m][0])); total += 1
    return ok / total

print()
print('LOPO accuracy (within-distribution):')
for name, _ in model_specs:
    pp = lopo_probs[name][orig_mask]
    print(f'  {name:<5}  pillar={pillar_acc(pp, y_orig, p_orig)*100:5.1f}%   '
          f'clip={((pp>=0.5).astype(int)==y_orig).mean()*100:5.1f}%')

ens_lopo = np.mean(np.stack([lopo_probs[n][orig_mask] for n,_ in model_specs]), axis=0)
print(f'  Ens   pillar={pillar_acc(ens_lopo, y_orig, p_orig)*100:5.1f}%   '
      f'clip={((ens_lopo>=0.5).astype(int)==y_orig).mean()*100:5.1f}%')

  10/31 folds in 59s


  20/31 folds in 117s


  30/31 folds in 177s



LOPO accuracy (within-distribution):
  SVM    pillar= 90.3%   clip= 81.5%
  GBM    pillar= 87.1%   clip= 79.1%
  RF     pillar= 93.5%   clip= 80.9%
  Ens   pillar= 87.1%   clip= 81.1%


## Step 2 — Final fit on all 31 training pillars

Now train each model on the entire augmented training set. SVM uses `probability=True` here (only one fit, can afford Platt's internal CV) for proper calibration.

In [6]:
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xho_s = scaler.transform(Xho)

final_models = []
for name, builder in model_specs:
    if name == 'SVM':
        clf = SVC(kernel='rbf', C=5.0, gamma=0.001, class_weight='balanced', probability=True)
    else:
        clf = builder()
    clf.fit(Xtr_s, ytr)
    final_models.append((name, clf))

# Per-model holdout probabilities
holdout_probs = {n: clf.predict_proba(Xho_s)[:, 1] for n, clf in final_models}
ens_holdout = np.mean(np.stack(list(holdout_probs.values())), axis=0)

print('Final fit complete.')
print(f'Train memorization check (ensemble): {((ens_holdout.mean() < 0.5) == (yho.mean() < 0.5))}')

Final fit complete.
Train memorization check (ensemble): True


## Step 3 — Holdout evaluation

Three modes:
1. **Default vote** (threshold 0.5) — all clips vote.
2. **Default + abstention** — clips with `P(bad) ∈ [0.4, 0.6]` don't vote. Pillar verdict is the majority of confident clips.
3. **Per-model breakdown** — sanity check that no model is dragging the ensemble down.

In [7]:
def eval_holdout(probs, threshold=0.5, abst_lo=None, abst_hi=None):
    """Returns (per_pillar_acc, per_clip_acc, n_pillars, n_correct, abst_clips, rows)."""
    rows = []
    n_p = n_p_ok = 0
    n_clip = n_clip_ok = abst_clips = 0
    for p in sorted(set(pho)):
        m = pho == p
        pr = probs[m]
        truth = int(yho[m][0])
        if abst_lo is not None:
            confident = (pr < abst_lo) | (pr > abst_hi)
            abst_clips += int((~confident).sum())
        else:
            confident = np.ones_like(pr, dtype=bool)
        if not confident.any():
            rows.append((p, truth, 'ABSTAIN', float(pr.mean()), len(pr), 0))
            continue
        votes = (pr[confident] >= threshold).astype(int)
        vote = int(np.bincount(votes, minlength=2).argmax())
        n_p += 1; n_p_ok += int(vote == truth)
        clip_correct = int(((pr >= threshold).astype(int) == truth).sum())
        n_clip += len(pr); n_clip_ok += clip_correct
        rows.append((p, truth, vote, float(pr.mean()), len(pr), clip_correct))
    return (n_p_ok/n_p if n_p else 0.0,
            n_clip_ok/n_clip if n_clip else 0.0,
            n_p, n_p_ok, abst_clips, rows)

def print_table(label, rows):
    print(f'\n{label}')
    print(f'{"pillar":<10}{"truth":<7}{"vote":<10}{"clips":>7}{"correct":>10}{"P(bad)":>10}')
    for p, truth, vote, pmean, nc, cc in rows:
        truth_s = 'BAD' if truth == 1 else 'GOOD'
        if vote == 'ABSTAIN':
            vote_s, mark = 'ABSTAIN', '·'
        else:
            vote_s = 'BAD' if vote == 1 else 'GOOD'
            mark = '✓' if vote == truth else '✗'
        # strip trailing m4a artifact from name
        name = ('good_' if p.startswith('good') else 'bad_') + p.split('_')[1][:-1]
        print(f'{name:<10}{truth_s:<7}{vote_s:<10}{nc:>7}{cc:>5}/{nc:<3}{pmean*100:>9.1f}%  {mark}')

print('=' * 70)
print('HOLDOUT — 6 unseen pillars')
print('=' * 70)

# Mode 1: default vote
p_acc, c_acc, n_p, n_p_ok, _, rows = eval_holdout(ens_holdout)
print_table(f'Mode 1 — Ensemble vote (threshold 0.5)', rows)
print(f'  Per-pillar: {n_p_ok}/{n_p} = {p_acc*100:.1f}%')
print(f'  Per-clip  : {((ens_holdout >= 0.5).astype(int) == yho).sum()}/{len(yho)} = {c_acc*100:.1f}%')

# Mode 2: with abstention
p_acc_a, c_acc_a, n_p_a, n_p_ok_a, abst, rows_a = eval_holdout(
    ens_holdout, abst_lo=ABSTENTION_LO, abst_hi=ABSTENTION_HI)
print_table(f'Mode 2 — Ensemble + abstention zone [{ABSTENTION_LO}, {ABSTENTION_HI}]', rows_a)
print(f'  Pillars decided when answered: {n_p_a}/{len(set(pho))}')
print(f'  Accuracy when answered       : {n_p_ok_a}/{n_p_a} = {p_acc_a*100:.1f}%')
print(f'  Clips abstained              : {abst}/{len(yho)} ({abst/len(yho)*100:.1f}%)')

# Mode 3: per-model breakdown
print('\n' + '-' * 70)
print('Per-model breakdown (no abstention):')
for name in [n for n,_ in model_specs] + ['Ensemble']:
    pp = ens_holdout if name == 'Ensemble' else holdout_probs[name]
    pa, ca, _, _, _, _ = eval_holdout(pp)
    pbg = pp[yho == 0].mean() * 100
    pbb = pp[yho == 1].mean() * 100
    print(f'  {name:<10}  pillar={pa*100:5.1f}%   clip={ca*100:5.1f}%   '
          f'P(bad) on GOOD={pbg:5.1f}%   on BAD={pbb:5.1f}%')

HOLDOUT — 6 unseen pillars

Mode 1 — Ensemble vote (threshold 0.5)
pillar    truth  vote        clips   correct    P(bad)
bad_19    BAD    BAD            11   11/11      92.1%  ✓
bad_20    BAD    BAD            20   19/20      88.9%  ✓
bad_21    BAD    GOOD           10    5/10      58.2%  ✗
good_14   GOOD   GOOD            7    7/7       17.2%  ✓
good_15   GOOD   GOOD           14   14/14      14.4%  ✓
good_16   GOOD   GOOD           10   10/10      10.1%  ✓
  Per-pillar: 5/6 = 83.3%
  Per-clip  : 66/72 = 91.7%

Mode 2 — Ensemble + abstention zone [0.4, 0.6]
pillar    truth  vote        clips   correct    P(bad)
bad_19    BAD    BAD            11   11/11      92.1%  ✓
bad_20    BAD    BAD            20   19/20      88.9%  ✓
bad_21    BAD    BAD            10    5/10      58.2%  ✓
good_14   GOOD   GOOD            7    7/7       17.2%  ✓
good_15   GOOD   GOOD           14   14/14      14.4%  ✓
good_16   GOOD   GOOD           10   10/10      10.1%  ✓
  Pillars decided when answered: 6/6


## Result summary

The numbers below are the headline metrics for the paper.

In [8]:
print('=' * 70)
print('FINAL HEADLINE NUMBERS')
print('=' * 70)
lopo_ens = pillar_acc(ens_lopo, y_orig, p_orig)
print(f'  LOPO accuracy on 31 training pillars (within-distribution): {lopo_ens*100:.1f}%')
print(f'  Holdout accuracy on 6 unseen pillars (default 0.5)        : {p_acc*100:.1f}%')
print(f'  Holdout accuracy with abstention [0.40, 0.60]             : {p_acc_a*100:.1f}%')
print(f'  Per-clip accuracy on holdout                              : {c_acc*100:.1f}%')
print(f'  Mean P(bad) on holdout GOOD pillars                       : {ens_holdout[yho==0].mean()*100:.1f}%')
print(f'  Mean P(bad) on holdout BAD  pillars                       : {ens_holdout[yho==1].mean()*100:.1f}%')
print()
print('Compare to original SVM on the same 6 holdout pillars:')
print('  Holdout per-pillar : 50.0%  (3/6 — all 3 good pillars wrongly called bad)')
print('  Holdout per-clip   : 56.9%')

FINAL HEADLINE NUMBERS
  LOPO accuracy on 31 training pillars (within-distribution): 87.1%
  Holdout accuracy on 6 unseen pillars (default 0.5)        : 83.3%
  Holdout accuracy with abstention [0.40, 0.60]             : 100.0%
  Per-clip accuracy on holdout                              : 91.7%
  Mean P(bad) on holdout GOOD pillars                       : 13.6%
  Mean P(bad) on holdout BAD  pillars                       : 82.3%

Compare to original SVM on the same 6 holdout pillars:
  Holdout per-pillar : 50.0%  (3/6 — all 3 good pillars wrongly called bad)
  Holdout per-clip   : 56.9%
